# Notebook 03: GroupBy Analysis – Summarizing EHR Data

In this notebook, you’ll practice summarizing healthcare data using both SQL and pandas:

- Grouping and aggregating patient and encounter data
- Calculating averages, counts, and maximums
- Comparing SQL `GROUP BY` with pandas `groupby()`

In [ ]:
# Specify the disk drive path to the folder containing the Synthea CSV files
# You'll change the path on the next line to match your system setup
my_path = "/Users/spb24/Downloads/coherent-11-07-2022/csv/"

In [ ]:
import pandas
import sqlite3

# Load CSV files
patients = pandas.read_csv(my_path + "patients.csv")
encounters = pandas.read_csv(my_path + "encounters.csv")

✅ **Checkpoint**  
Run `patients.head()` and `encounters.head()` to review column names before grouping.

In [ ]:
# Create an in-memory SQLite database for querying
rdbms = sqlite3.connect(":memory:")

# Store dataframes as SQL tables
patients.to_sql("patients", rdbms, index=False, if_exists="replace")
encounters.to_sql("encounters", rdbms, index=False, if_exists="replace")

### 🔹 SQL: Count of Encounters by Encounter Class

In this SQL query, we perform a group-by operation to summarize the number of encounters by class:

- `GROUP BY ENCOUNTERCLASS` groups the rows by type of encounter (e.g., emergency, wellness)
- `COUNT(*)` counts how many rows are in each group
- `AS num_encounters` gives a readable name to the result column
- `ORDER BY num_encounters DESC` sorts the results by count, highest first

In [ ]:
# Write and execute SQL query to group by ENCOUNTERCLASS
query = """
SELECT ENCOUNTERCLASS, COUNT(*) AS num_encounters
FROM encounters
GROUP BY ENCOUNTERCLASS
ORDER BY num_encounters DESC;
"""

pandas.read_sql_query(query, rdbms)

### 🔁 pandas: Same Summary Using Python

This accomplishes the same as the SQL query above. Let's break it down:

- `.groupby("ENCOUNTERCLASS")` groups the data
- `.size()` counts rows in each group
- `.sort_values()` puts most common first
- `.reset_index(name=...)` converts result back to DataFrame and names the count column

In [ ]:
(encounters.groupby("ENCOUNTERCLASS").size()
    .sort_values(ascending=False)
    .reset_index(name="num_encounters"))

### 🔹 SQL: Average Age by Gender

To compute age, we first calculate it in the dataframe and write it back into the SQLite database. Then we can group by gender.

In [ ]:
# Prepare age column and write back to SQL
from datetime import datetime

patients['BIRTHDATE'] = pandas.to_datetime(patients['BIRTHDATE'], errors='coerce')
patients['age'] = (datetime.today() - patients['BIRTHDATE']).dt.days // 365
patients.to_sql("patients", rdbms, index=False, if_exists="replace")

# SQL query to get average age by gender
query = """
SELECT GENDER, AVG(age) AS avg_age
FROM patients
GROUP BY GENDER;
"""
pandas.read_sql_query(query, rdbms)

✏️ **Reflection**  
Which approach (SQL or pandas) did you find more intuitive for these summaries? Why?